# ⚠️ mova **판단 단계(도구 호출)** LoRA — v10 · Google Colab

> **이해 노트북(v8 `understand_…`)·추천 노트북(`mova_…`)이 아닙니다.** 산출물 이름이 `agent_…`로 시작하고
> 리포트는 `report_agent-v10.json`입니다.

채팅 발화(+최근 대화+이번 턴의 도구 결과)를 읽고 **다음 행동 하나** — 도구 호출 1개(`<tool_call>{…}</tool_call>`)
또는 `FINAL` — 를 내는 모델을 EXAONE-3.5-**2.4B** LoRA로 학습한다. 사실 문장은 모델이 쓰지 않는다(코드 템플릿).

    <tool_call>{"name":"showtimes","arguments":{"title":"인턴","region":"군자","date":"오늘"}}</tool_call>

| 단계 | 내용 | 예상 시간(L4) |
|---|---|---|
| 1~3 | 설치 · 드라이브 · 데이터 · 템플릿 일치 검사 | 3분 |
| 4 | 학습 전 2.4B 채점(평가셋 66) | 1분 |
| 5 | 학습(에폭마다 검증 손실) → 최적 에폭 채점 | 15~25분 |
| 6 | 판단 리포트 | — |
| 7 | 학습+검증 전체 재학습 → 병합 → GGUF(Q5_K_M) → Ollama Modelfile | 15~20분 |
| 8 | 산출물 내려받기 | — |

**교사 모델·API 키가 필요 없다.** 라벨은 데이터셋을 만든 템플릿의 행동 그대로다.

### 시작 전 준비
1. **런타임 → 런타임 유형 변경 → GPU: L4 또는 A100**
2. 바탕화면 `mova/FT/agent-v10/`의 jsonl 세 개를 드라이브 `qwen-training/agent-v10/`에 올려 두거나,
   3번 셀이 띄우는 업로드 창에서 세 개를 한꺼번에 고른다.
3. **런타임 → 모두 실행**. 드라이브 연결이 실패해도 끝까지 진행되고 8번 셀에서 내려받는다.

### 합격 기준(평가셋 75 = 멀티턴 17 + 단일 28 + 09-29 실사용 대화 17 + v10 신규 9 + 섀도 4, 사람이 단 정답)
| 모델(09-29 저녁 노트북 실측, GGUF) | 정답(도구+인자) |
|---|---|
| **v9 (2.4B 학습, 도구 6개 — 새 도구 4개는 모름)** | 71/75 |
| 운영 7.8B(프롬프트만, 도구 10개) | 69/75 |
| 학습 전 2.4B(프롬프트만) | 59/75 |

채점은 v9 노트북과 같다(`scripts/build_agent_dataset.py` `match`). 학습한 2.4B가 **전체 72/75 이상**이고
**실사용 대화(live) 14/17 이상·v10 신규 9/9·형식 실패 0**이면 7번 셀로 내보낸다. 최종 판단은 노트북 GGUF 재채점
(`scripts/eval_agent_actions.py mova-agent-v10`).

### v10에서 바뀐 것(v9 대비) — 2026-09-29 운영 전환 뒤 실측
- **도구 4개 추가**: `recommend_for_me`(취향: "내 취향", 맨 "추천해줘") · `similar_to`("○○ 같은 영화") · `mark_watched`
  ("그거 봤어, 4점") · `taste_profile`("내 취향이 뭐야"). 조건 추천과 취향 추천의 경계 규칙을 프롬프트에 명시.
- **운영 약점 3개**: 지시어는 가장 최근 작품(`referent_recent`), 조사 붙은 제목("데자뷰는 누가 나오지", `cast_particle`),
  시리즈 이름은 부제 완성 금지(규칙). 날짜 후속은 date 포함(규칙).
- 시스템 프롬프트가 v9와 다르다 — 서빙 `agent_prompt.py`와 같은 파일에서 생성했으므로 그대로 배포하면 일치.

### v9가 v7/v8(이해 단계)과 다른 것
- 출력이 6칸 JSON이 아니라 **행동 하나**. 한 턴에 도구 하나씩, 최대 3단계(초과는 코드가 FINAL 강제).
- 입력에 `[도구 결과]` 절이 붙는 2단계 행(208행)이 있다 — 결과를 받고 FINAL 하거나 다음 도구를 고른다.
- 09-29 실사용 오답이 전부 패턴이다: 출연진("누가 나와"→details), 현재 상영작(→now_showing), 시리즈 신작
  (→search_movie), 직전 카드 전체 후속("다 영화관에서"→now_showing), 지시어·주제 전환.
- 가드(제목·지역 근거, 호출 예산)는 학습이 아니라 서빙 코드가 담당한다.


In [ ]:
# 1) GPU 확인 + 라이브러리 — v5 노트북과 같은 고정 버전
# transformers는 5.5.0 고정 — EXAONE 원격 코드가 4.57은 import 실패, 5.13은 create_causal_mask 인자 불일치(09-17 실측)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip uninstall -y -q torchao
!pip install -q "transformers==5.5.0" "peft==0.20.0" accelerate sentencepiece

In [ ]:
# 2) 설정
MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"   # 운영 Ollama exaone3.5:2.4b와 같은 베이스
DRIVE_DIR = "/content/drive/MyDrive/qwen-training"
VERSION_TAG = "agent-v10"
DATA_SUBDIR = "agent-v10"
FILES = ["agent_train.jsonl", "agent_val.jsonl", "agent_eval.jsonl"]
ADAPTER_DIR = "/content/adapters"

EPOCHS = 3            # 최대 에폭. 검증 손실이 가장 낮은 에폭을 자동 선택
LR = 1e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj", "c_fc_0", "c_fc_1", "c_proj"]  # EXAONE 선형층 이름
MAX_NEW_TOKENS = 120  # <tool_call>{…}</tool_call> 한 줄 또는 FINAL
FULL_RETRAIN = True   # 평가 후 train+val 전체로 다시 학습해 내보낼지(평가셋은 끝까지 안 섞는다)
QUANT = "Q5_K_M"
LLAMA_CPP_COMMIT = "304665fe7ac957df95e3ff8c8c4ffdf92dd6ffa3"  # 노트북 서빙 llama.cpp와 동일 커밋
SEED = 20260930
PASS_ALL, PASS_LIVE = 72, 14   # v9 GGUF(71/75)·7.8B(69/75)를 넘어야 의미가 있다
BASE_78B, BASE_12B = 69, 71

import torch
DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
EXPORT_DTYPE = torch.float16

In [ ]:
# 3) 드라이브 연결 + 데이터셋 — 실패해도 로컬로 진행(v5와 같은 폴백)
import os, shutil, json, random, time
from datetime import datetime, timezone
from google.colab import drive, files

DRIVE_OK = False
for _try in range(2):
    try:
        drive.mount("/content/drive", force_remount=bool(_try))
        DRIVE_OK = True
        break
    except Exception as e:
        print(f"드라이브 연결 실패 {_try + 1}/2: {e}")
if not DRIVE_OK:
    DRIVE_DIR = "/content/qwen-training"
    print(f"⚠ 드라이브 없이 진행 — 산출물은 {DRIVE_DIR}/out/{VERSION_TAG}, 8번 셀에서 내려받기")
DATA_DIR = f"{DRIVE_DIR}/{DATA_SUBDIR}"
OUT_DIR = f"{DRIVE_DIR}/out/{VERSION_TAG}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

missing = [f for f in FILES if not os.path.exists(f"{DATA_DIR}/{f}")]
if missing:
    print(f"없는 파일 {missing} — 업로드 창에서 바탕화면 mova/FT/agent-v10/의 jsonl을 모두 고르세요")
    for name in files.upload():
        shutil.move(name, f"{DATA_DIR}/{os.path.basename(name)}")
missing = [f for f in FILES if not os.path.exists(f"{DATA_DIR}/{f}")]
assert not missing, f"여전히 없음: {missing}"

def load(name):
    return [json.loads(l) for l in open(f"{DATA_DIR}/{name}", encoding="utf-8") if l.strip()]

train_rows, val_rows, eval_rows = (load(f) for f in FILES)
print(f"train {len(train_rows)} · val {len(val_rows)} · eval {len(eval_rows)}")
assert len({r["system"] for r in train_rows + val_rows + eval_rows}) == 1, "system 프롬프트가 행마다 다름"
# 추천 데이터(chat_teacher_dataset_*)를 잘못 올리면 여기서 멈춘다(09-28: 노트북이 섞였던 일)
assert all("gold" in r for r in train_rows[:5]) and "판단 담당" in train_rows[0]["system"], (
    "판단 단계 데이터가 아닙니다 — agent_*.jsonl 세 개를 올리세요"
)
SYSTEM = train_rows[0]["system"]

In [ ]:
# 3-1) 모델·토크나이저 + 서빙 템플릿 일치 검사
# 운영은 Ollama(`understand_json` → system+prompt)로 서빙한다. Ollama EXAONE 템플릿이 만드는 문자열과
# 학습 입력(HF chat template)이 다르면 학습한 분포와 서빙 분포가 어긋난다 — 여기서 글자 단위로 확인한다.
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def load_base(dtype=None):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=dtype or DTYPE, device_map=DEVICE, trust_remote_code=True
    )
    type(model.transformer).get_input_embeddings = lambda self: self.wte  # peft·gc가 호출
    return model

def messages(r):
    return [{"role": "system", "content": r["system"]}, {"role": "user", "content": r["prompt"]}]

def chat_ids(r):
    return tok.apply_chat_template(
        messages(r), add_generation_prompt=True, return_tensors="pt", return_dict=True
    )["input_ids"]

def ollama_render(system, prompt):
    # `ollama show exaone3.5:2.4b --modelfile`의 TEMPLATE을 그대로 옮긴 것(09-28)
    return f"[|system|]{system}[|endofturn|]\n[|user|]{prompt}\n[|assistant|]"

_hf = tok.apply_chat_template(messages(eval_rows[0]), add_generation_prompt=True, tokenize=False)
_ol = ollama_render(eval_rows[0]["system"], eval_rows[0]["prompt"])
assert _hf.endswith(_ol) and len(_hf) - len(_ol) < 8, f"템플릿 불일치!\nHF: {_hf[-200:]!r}\nOL: {_ol[-200:]!r}"
print("✓ HF 학습 템플릿 == Ollama 서빙 템플릿")

_p = tok.apply_chat_template(messages(eval_rows[0]), add_generation_prompt=True, tokenize=False)
_f = tok.apply_chat_template(messages(eval_rows[0]) + [{"role": "assistant", "content": "Y"}], tokenize=False)
assert _f.startswith(_p + "Y"), "chat template 구조가 예상과 다름"
TURN_END = _f[len(_p) + 1 :]

@torch.no_grad()
def generate(model, r):
    model.eval()
    ids = chat_ids(r).to(model.device)
    out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
                         max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# 3-2) 채점 — 도구 이름 + 정답 인자 정규화 일치(scripts/build_agent_dataset.py의 match와 같은 함수)
import re
TOOLS = {"search_movie": ("title",), "get_movie_details": ("title",), "recommend_movies": ("query",),
         "now_showing": (), "showtimes": ("title", "region", "date"), "where_to_watch": ("title",)}
_CALL = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def parse_action(text):
    m = _CALL.search(text or "")
    if m:
        try:
            data = json.loads(m.group(1))
        except json.JSONDecodeError:
            return None
        name = data.get("name")
        if name not in TOOLS:
            return None
        raw = data.get("arguments") or {}
        args = {k: str(v).strip() for k, v in raw.items() if k in TOOLS[name] and isinstance(v, (str, int, float)) and str(v).strip()}
        return {"name": name, "arguments": args}
    return "FINAL" if re.search(r"\bFINAL\b", text or "") else None

def norm(s):
    return re.sub(r"[\s『』'\"():.!?~,·]", "", s or "").lower()

def norm_region(s):
    return re.sub(r"(근처|쪽|에서|에|으로|로|일대|전체|전역)$", "", norm(s))

def match(pred, gold):
    if gold == "FINAL" or pred == "FINAL" or pred is None:
        return pred == gold
    if not isinstance(pred, dict) or pred.get("name") != gold["name"]:
        return False
    ga, pa = gold["arguments"], pred.get("arguments") or {}
    for k, v in ga.items():
        p = pa.get(k, "")
        if k == "query":
            if not p:
                return False
        elif k == "region":
            if norm_region(p) != norm_region(v):
                return False
        elif norm(p) != norm(v) and not (k == "title" and norm(v) and norm(v) in norm(p)):
            return False
    return all(k in ga for k in ("title", "region") if k in pa)

def score(outputs, rows, label):
    n = len(rows)
    ok, bad_format, fails = 0, 0, []
    by_src = {}
    for raw, r in zip(outputs, rows):
        pred = parse_action(raw)
        hit = match(pred, r["gold"])
        src = r["pattern"].split(":")[0]
        by_src.setdefault(src, [0, 0])
        by_src[src][1] += 1
        if pred is None:
            bad_format += 1
        if hit:
            ok += 1
            by_src[src][0] += 1
        else:
            fails.append((r, raw))
    live = by_src.get("live", [0, 0])
    line = f"{label:12} 정답 {ok}/{n} · " + " ".join(f"{s} {a}/{b}" for s, (a, b) in sorted(by_src.items())) + f" · 형식 실패 {bad_format}"
    print(line)
    return {"all": ok, "n": n, "live": live[0], "by_src": by_src, "bad_format": bad_format, "fails": fails, "line": line}

def show_fails(res, k=10):
    for r, raw in res["fails"][:k]:
        msg = r["prompt"].split("[발화]\n")[-1].split("\n")[0][:40]
        print(f"  ✗ {msg!r}\n     예측 {raw.strip()[:140]}\n     정답 {r['completion']}")


## 4. 학습 전 2.4B 채점 — 노트북 실측(59/75)과 비슷하면 파이프라인이 운영과 같은 조건이다


In [ ]:
base = load_base()
t0 = time.time()
base_out = [generate(base, r) for r in eval_rows]
print(f"({time.time() - t0:.0f}s)")
base_res = score(base_out, eval_rows, "base 2.4B")
show_fails(base_res, 5)
del base; free()

## 5. 학습 — 에폭마다 검증 손실, 최적 에폭 어댑터로 평가셋 채점

In [ ]:
def example(r):
    # 프롬프트와 답을 따로 토큰화해 잇는다(합치면 경계 토큰이 달라져 추론과 어긋난다, 09-17 실측)
    p = chat_ids(r)
    c = tok(r["completion"] + TURN_END, add_special_tokens=False, return_tensors="pt")["input_ids"]
    f = torch.cat([p, c], dim=1)
    labels = f.clone()
    labels[:, : p.shape[1]] = -100   # 프롬프트는 손실 제외, JSON만 학습
    return f, labels

def train(train_set, epochs, eval_set=None, tag="run"):
    model = get_peft_model(load_base(), LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGETS, task_type="CAUSAL_LM"))
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.print_trainable_parameters()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    data = [example(r) for r in train_set]
    eval_data = [example(r) for r in (eval_set or [])]
    order = random.Random(SEED)
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        order.shuffle(data)
        total, t0 = 0.0, time.time()
        for i, (ids, labels) in enumerate(data, 1):
            loss = model(input_ids=ids.to(model.device), labels=labels.to(model.device), use_cache=False).loss
            loss.backward()
            opt.step()
            opt.zero_grad()
            total += loss.item()
            if i % 100 == 0 or i == len(data):
                print(f"  epoch {ep} {i}/{len(data)} loss={total / i:.4f} ({time.time() - t0:.0f}s)")
        row = {"epoch": ep, "train_loss": round(total / len(data), 4)}
        if eval_data:
            model.eval()
            with torch.no_grad():
                row["val_loss"] = round(sum(
                    model(input_ids=i_.to(model.device), labels=l_.to(model.device), use_cache=False).loss.item()
                    for i_, l_ in eval_data) / len(eval_data), 4)
        model.save_pretrained(f"{ADAPTER_DIR}/{tag}_ep{ep}")
        history.append(row)
        print(row)
    return model, history

In [ ]:
model, history = train(train_rows, EPOCHS, val_rows, tag="eval")
del model; free()
BEST_EPOCH = min(history, key=lambda h: h["val_loss"])["epoch"]
print(f"최적 에폭: {BEST_EPOCH}")

student = PeftModel.from_pretrained(load_base(), f"{ADAPTER_DIR}/eval_ep{BEST_EPOCH}")
t0 = time.time()
student_out = [generate(student, r) for r in eval_rows]
print(f"평가셋 {len(eval_rows)}건 {time.time() - t0:.1f}s")
student_res = score(student_out, eval_rows, f"학생 ep{BEST_EPOCH}")
show_fails(student_res)
# 검증셋 표본으로 템플릿 분포 안에서의 정확도도 본다(평가셋은 분포 밖)
val_sample = random.Random(SEED).sample(val_rows, min(80, len(val_rows)))
val_res = score([generate(student, r) for r in val_sample], val_sample, "학생(val80)")
del student; free()

## 6. 판단 리포트

In [ ]:
print(f"■ 평가셋(멀티턴 17 + 단일 28 + 실사용 17 + 섀도 {len(eval_rows)}개) — 도구+인자 정답")
print(f"  운영 7.8B {BASE_78B}/75 · v9 {BASE_12B}/75 (09-29 저녁 GGUF 실측)")
print("  " + base_res["line"])
print("  " + student_res["line"])
v10_ok = student_res["by_src"].get("v10", [0, 0])[0] >= 9  # 새 도구 4개 평가 9건 전부
passed = (student_res["all"] >= PASS_ALL and student_res["live"] >= PASS_LIVE and v10_ok and student_res["bad_format"] == 0)
print("\n판정:", "✅ v9 초과 — 7번 셀로 내보내기" if passed else "❌ 기준 미달 — 실패 사례를 보고 데이터셋을 보강")

report = {
    "version_tag": VERSION_TAG, "best_epoch": BEST_EPOCH, "history": history, "passed": passed,
    "baseline_78b_all": BASE_78B, "baseline_v9_all": BASE_12B,
    "base": {k: v for k, v in base_res.items() if k != "fails"},
    "student": {k: v for k, v in student_res.items() if k != "fails"},
    "student_val80": {k: v for k, v in val_res.items() if k != "fails"},
    "student_fails": [{"prompt": r["prompt"][-300:], "gold": r["completion"], "pred": raw} for r, raw in student_res["fails"]],
}
with open(f"{OUT_DIR}/report_{VERSION_TAG}.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"리포트: {OUT_DIR}/report_{VERSION_TAG}.json")


### 기준 미달일 때
- `title`이 낮으면: 조사 붙은 제목·콜론 제목 패턴을 늘린다(`build_understanding_dataset.py` 제목 템플릿).
- `followup`이 낮으면: 후속 패턴(지역 답·날짜 답·지시어) 비율을 올린다.
- 평가셋 36개는 **학습에 넣지 않는다** — 넣으면 합격 기준이 무의미해진다.

## 7. 전체 재학습 → 병합 → GGUF → Ollama Modelfile

In [ ]:
import subprocess, sys

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

VERSION = "agent_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
if FULL_RETRAIN:
    model, _ = train(train_rows + val_rows, BEST_EPOCH, tag="full")
    del model; free()
    adapter = f"{ADAPTER_DIR}/full_ep{BEST_EPOCH}"
else:
    adapter = f"{ADAPTER_DIR}/eval_ep{BEST_EPOCH}"
shutil.copytree(adapter, f"{OUT_DIR}/{VERSION}-adapter", dirs_exist_ok=True)

MERGED = f"/content/{VERSION}-merged"
merged = PeftModel.from_pretrained(load_base(EXPORT_DTYPE), adapter).merge_and_unload()
merged.save_pretrained(MERGED, max_shard_size="1GB")
tok.save_pretrained(MERGED)
del merged; free()

LLAMA = "/content/llama.cpp"
if not os.path.exists(f"{LLAMA}/build/bin/llama-quantize"):
    sh(f"rm -rf {LLAMA} && git init -q {LLAMA}")
    sh(f"cd {LLAMA} && git fetch -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_COMMIT} && git checkout -q FETCH_HEAD")
    sh(f"pip install -q -r {LLAMA}/requirements/requirements-convert_hf_to_gguf.txt || true")
    sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null")
    sh(f"cmake --build {LLAMA}/build --target llama-quantize -j{os.cpu_count()} > /dev/null")

F16 = f"/content/{VERSION}-f16.gguf"
OUT_GGUF = f"/content/{VERSION}-{QUANT}.gguf"
# llama.cpp 304665f ExaoneModel이 layer_norm_rms_epsilon을 안 써서 quantize가 죽는다 — 한 줄 보충(v5와 동일)
_patch = f"""
import runpy, sys
llama = {LLAMA!r}
sys.path[:0] = [llama, llama + "/gguf-py"]
from conversion import exaone
_orig = exaone.ExaoneModel.set_gguf_parameters
def _p(self):
    _orig(self)
    self.gguf_writer.add_layer_norm_rms_eps(self.hparams["layer_norm_epsilon"])
exaone.ExaoneModel.set_gguf_parameters = _p
sys.argv = [llama + "/convert_hf_to_gguf.py", {MERGED!r}, "--outfile", {F16!r}, "--outtype", "f16"]
runpy.run_path(sys.argv[0], run_name="__main__")
"""
with open("/content/convert_exaone.py", "w") as f:
    f.write(_patch)
sh(f"{sys.executable} /content/convert_exaone.py")
sh(f"{LLAMA}/build/bin/llama-quantize {F16} {OUT_GGUF} {QUANT}")
shutil.copy(OUT_GGUF, OUT_DIR)

# Ollama Modelfile — 운영 exaone3.5:2.4b와 같은 템플릿. temperature 0(결정론), stop 토큰 동일.
TEMPLATE_LINES = [
    "{{- range $i, $_ := .Messages }}",
    "{{- $last := eq (len (slice $.Messages $i)) 1 -}}",
    '{{ if eq .Role "system" }}[|system|]{{ .Content }}[|endofturn|]',
    "{{ continue }}",
    '{{ else if eq .Role "user" }}[|user|]{{ .Content }}',
    '{{ else if eq .Role "assistant" }}[|assistant|]{{ .Content }}[|endofturn|]',
    "{{ end }}",
    '{{- if and (ne .Role "assistant") $last }}[|assistant|]{{ end }}',
    "{{- end -}}",
]
Q3 = '"' * 3
MODELFILE = "\n".join([
    f"FROM ./{os.path.basename(OUT_GGUF)}",
    "TEMPLATE " + Q3 + "\n".join(TEMPLATE_LINES) + Q3,
    "PARAMETER temperature 0",
    "PARAMETER stop [|endofturn|]",
    "",
])
with open(f"{OUT_DIR}/Modelfile", "w") as f:
    f.write(MODELFILE)
print(MODELFILE)
print(f"완료: {OUT_DIR}/{os.path.basename(OUT_GGUF)} ({os.path.getsize(OUT_GGUF) / 1e9:.2f} GB) + Modelfile")

## 8. 산출물 내려받기 + 노트북(teagy)에서 GGUF 재채점

드라이브에 저장됐으면 `내 드라이브/qwen-training/out/agent-v10/`에서 GGUF와 `Modelfile`을 받는다
(입력 폴더 `agent-v10`가 아니라 **출력 폴더 `out/agent-v10`**).

노트북(teagy)에서:

    mkdir -p ~/models/mova-agent-v10 && cd ~/models/mova-agent-v10   # GGUF·Modelfile·report 복사
    echo "PARAMETER num_gpu 0" >> Modelfile                          # CPU 전용(운영 GPU 보호)
    ollama create mova-agent-v10 -f Modelfile
    cd ~/projects/suvisdev/suvisdev && ~/.venv-exaone/bin/python scripts/eval_agent_actions.py mova-agent-v10 --cpu

GGUF 채점이 7.8B를 넘으면 서빙 어댑터(판단 단계 + 가드 + 사실 템플릿) 작업으로 넘어간다.


In [ ]:
import glob, zipfile

print(f"산출물 폴더: {OUT_DIR}")
for p in sorted(glob.glob(f"{OUT_DIR}/*")):
    size = f"{os.path.getsize(p) / 1e6:.1f} MB" if os.path.isfile(p) else "(폴더)"
    print(f"  {os.path.basename(p)}  {size}")
if DRIVE_OK:
    print(f"드라이브에 저장돼 있습니다 — 내 드라이브/qwen-training/out/{VERSION_TAG}/")
else:
    small = f"/content/{VERSION_TAG}-adapter-report-modelfile.zip"
    with zipfile.ZipFile(small, "w", zipfile.ZIP_DEFLATED) as z:
        for p in glob.glob(f"{OUT_DIR}/**/*", recursive=True):
            if os.path.isfile(p) and not p.endswith(".gguf"):
                z.write(p, os.path.relpath(p, OUT_DIR))
    files.download(small)
    for g in glob.glob(f"{OUT_DIR}/*.gguf"):
        print(f"GGUF 내려받기: {g} ({os.path.getsize(g) / 1e9:.2f} GB)")
        files.download(g)